In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Ketaksamaan CHSH

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Anggaran penggunaan: Dua minit pada pemproses Heron r2 (NOTA: Ini adalah anggaran sahaja. Masa jalan sebenar mungkin berbeza.)*
## Latar belakang
Dalam tutorial ini, kamu akan menjalankan eksperimen pada komputer kuantum untuk menunjukkan pelanggaran ketaksamaan CHSH menggunakan primitif Estimator.

Ketaksamaan CHSH, dinamakan sempena pengarang-pengarangnya Clauser, Horne, Shimony, dan Holt, digunakan untuk membuktikan teorem Bell secara eksperimental (1969). Teorem ini menyatakan bahawa teori pembolehubah tersembunyi setempat tidak dapat menjelaskan beberapa kesan keterjaitan dalam mekanik kuantum. Pelanggaran ketaksamaan CHSH digunakan untuk menunjukkan bahawa mekanik kuantum tidak serasi dengan teori pembolehubah tersembunyi setempat. Ini adalah eksperimen penting untuk memahami asas mekanik kuantum.

Hadiah Nobel Fizik 2022 telah dianugerahkan kepada Alain Aspect, John Clauser dan Anton Zeilinger sebahagiannya atas kerja perintis mereka dalam sains maklumat kuantum, khususnya eksperimen mereka dengan foton yang terjalin yang menunjukkan pelanggaran ketaksamaan Bell.
## Keperluan
Sebelum memulakan tutorial ini, pastikan kamu telah memasang yang berikut:

* Qiskit SDK v1.0 atau lebih baru, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 atau lebih baru
## Persediaan

In [1]:
# General
import numpy as np

# Qiskit imports
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Qiskit Runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

# Qiskit Aer for local noisy simulation
from qiskit_aer import AerSimulator

# Plotting routines
import matplotlib.pyplot as plt
import matplotlib.ticker as tck

In [ ]:
# Select an IBM Quantum backend.
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=127, operational=True, simulator=False
)
backend.name

'ibm_pittsburgh'

## Small-scale simulator example

Before submitting a hardware job, we validate the entire workflow on a local noisy simulator. We use `AerSimulator.from_backend(backend)` to build a simulator that inherits the noise model and coupling map of the backend you selected, so the simulator response is qualitatively similar to what we expect from hardware.

### Step 1: Map classical inputs to a quantum problem

We write the CHSH circuit with a single parameter $\theta$, which sweeps the measurement basis of the first qubit. The [`Estimator`](/docs/api/qiskit-ibm-runtime/estimator-v2) primitive simplifies the analysis: it returns expectation values of observables directly, and it can evaluate a parameterized circuit at many parameter values in a single call.

In [3]:
theta = Parameter(r"$\theta$")

chsh_circuit = QuantumCircuit(2)
chsh_circuit.h(0)
chsh_circuit.cx(0, 1)
chsh_circuit.ry(theta, 0)
chsh_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/c3f57d25-0.avif" alt="Output of the previous code cell" />

### Cipta Circuit CHSH berparameter
Pertama, kita tulis Circuit dengan parameter $\theta$, yang kita panggil `theta`. Primitif [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2) boleh memudahkan pembinaan Circuit dan analisis output dengan besar dengan menyediakan nilai jangkaan boleh ukur secara terus. Banyak masalah yang menarik, terutamanya untuk aplikasi jangka pendek pada sistem bising, boleh diformulasikan dalam sebutan nilai jangkaan. Primitif `Estimator` (V2) boleh menukar asas ukuran secara automatik berdasarkan boleh ukur yang diberikan.

In [4]:
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
# Phases need to be expressed as a list of lists for the Estimator PUB
individual_phases = [[ph] for ph in phases]

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/6c77e40a-0.avif)

### Cipta senarai nilai fasa untuk ditetapkan kemudian
Selepas mencipta Circuit CHSH berparameter, kamu akan mencipta senarai nilai fasa untuk ditetapkan pada Circuit dalam langkah seterusnya. Kamu boleh menggunakan kod berikut untuk mencipta senarai 21 nilai fasa yang merentangi $0$ hingga $2 \pi$ dengan jarak yang sama, iaitu $0$, $0.1 \pi$, $0.2 \pi$, ..., $1.9 \pi$, $2 \pi$.

In [5]:
# <S_1> = <ZZ> - <ZX> + <XZ> + <XX>
observable1 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)]
)

# <S_2> = <ZZ> + <ZX> - <XZ> + <XX>
observable2 = SparsePauliOp.from_list(
    [("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)]
)

### Boleh Ukur
Sekarang kita perlukan boleh ukur untuk mengira nilai jangkaan. Dalam kes kita, kita melihat asas ortogon bagi setiap qubit, membiarkan putaran $Y-$ berparameter bagi qubit pertama menyapu asas ukuran hampir berterusan berbanding asas qubit kedua. Oleh itu, kita akan memilih boleh ukur $ZZ$, $ZX$, $XZ$, dan $XX$.

In [6]:
# Build a noisy simulator from the ibm_pittsburgh backend
aer_sim = AerSimulator.from_backend(backend)

pm = generate_preset_pass_manager(target=aer_sim.target, optimization_level=3)
chsh_isa_circuit = pm.run(chsh_circuit)
chsh_isa_circuit.draw(output="mpl", idle_wires=False, style="iqp")

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/3e139a89-0.avif" alt="Output of the previous code cell" />

## Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Untuk mengurangkan jumlah masa pelaksanaan kerja, primitif V2 hanya menerima Circuit dan boleh ukur yang mematuhi arahan dan ketersambungan yang disokong oleh sistem sasaran (dirujuk sebagai Circuit dan boleh ukur ISA, iaitu instruction set architecture).

### Circuit ISA

In [7]:
isa_observable1 = observable1.apply_layout(layout=chsh_isa_circuit.layout)
isa_observable2 = observable2.apply_layout(layout=chsh_isa_circuit.layout)

![Output of the previous code cell](../docs/images/tutorials/chsh-inequality/extracted-outputs/9a5561eb-0.avif)

### Boleh Ukur ISA
Begitu juga, kita perlu mengubah boleh ukur supaya serasi dengan Backend sebelum menjalankan kerja dengan [`Runtime Estimator V2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run). Kita boleh melakukan transformasi menggunakan kaedah `apply_layout` pada objek `SparsePauliOp`.

In [8]:
# Use the AerSimulator-backed Estimator to validate the workflow locally
estimator_sim = Estimator(mode=aer_sim)

pub = (
    chsh_isa_circuit,  # ISA circuit
    [[isa_observable1], [isa_observable2]],  # ISA observables
    individual_phases,  # Parameter values
)

sim_result = estimator_sim.run(pubs=[pub]).result()

## Langkah 3: Laksanakan menggunakan primitif Qiskit
Untuk melaksanakan keseluruhan eksperimen dalam satu panggilan kepada [`Estimator`](https://docs.quantum-computing.ibm.com/api/qiskit-ibm-runtime/qiskit_ibm_runtime.EstimatorV2).
Kita boleh mencipta primitif [Qiskit Runtime `Estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) untuk mengira nilai jangkaan kita. Kaedah `EstimatorV2.run()` mengambil iterable bagi `primitive unified blocs (PUB)`. Setiap PUB adalah iterable dalam format `(circuit, observables, parameter_values: Optional, precision: Optional)`.

In [9]:
chsh1_sim = sim_result[0].data.evs[0]
chsh2_sim = sim_result[0].data.evs[1]


def plot_chsh(phases, chsh1, chsh2, title):
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(
        phases / np.pi, chsh1, "o-", label=r"$\langle S_1 \rangle$", zorder=3
    )
    ax.plot(
        phases / np.pi, chsh2, "o-", label=r"$\langle S_2 \rangle$", zorder=3
    )

    # classical bound +-2
    ax.axhline(y=2, color="0.9", linestyle="--")
    ax.axhline(y=-2, color="0.9", linestyle="--")

    # quantum bound, +-2*sqrt(2)
    ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
    ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
    ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
    ax.fill_between(
        phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7
    )

    ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
    ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel("CHSH witness")
    ax.set_title(title)
    ax.legend()
    plt.show()


plot_chsh(
    phases,
    chsh1_sim,
    chsh2_sim,
    "CHSH witnesses from AerSimulator (ibm_pittsburgh noise model)",
)

<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/c8fd5140-0.avif" alt="Output of the previous code cell" />

## Langkah 4: Proses pasca dan kembalikan hasil dalam format klasik yang dikehendaki
Estimator memulangkan nilai jangkaan bagi kedua-dua boleh ukur, $\langle ZZ \rangle - \langle ZX \rangle + \langle XZ \rangle + \langle XX \rangle$ dan $\langle ZZ \rangle + \langle ZX \rangle - \langle XZ \rangle + \langle XX \rangle$.

In [10]:
# -------------------------Step 1: Map classical inputs to a quantum problem-------------------------
# A CHSH test is bipartite, so we scale up by running one independent CHSH
# experiment on every disjoint Bell pair the device can host. A greedy
# matching of the coupling map gives a set of edges that share no qubits.
num_qubits = backend.num_qubits
used = set()
pairs = []
for qa, qb in backend.coupling_map.get_edges():
    if qa not in used and qb not in used:
        pairs.append((qa, qb))
        used.update((qa, qb))
num_pairs = len(pairs)
print(
    f"Tiling {backend.name} with {num_pairs} parallel Bell pairs "
    f"({2 * num_pairs} of {num_qubits} qubits)"
)

# One parameterized CHSH sub-circuit per pair, all sharing the angle theta
theta = Parameter(r"$\theta$")
chsh_circuit = QuantumCircuit(num_qubits)
for qa, qb in pairs:
    chsh_circuit.h(qa)
    chsh_circuit.cx(qa, qb)
    chsh_circuit.ry(theta, qa)

# Embed the two CHSH observables onto each pair's qubits (identity elsewhere)
obs1 = SparsePauliOp.from_list([("ZZ", 1), ("ZX", -1), ("XZ", 1), ("XX", 1)])
obs2 = SparsePauliOp.from_list([("ZZ", 1), ("ZX", 1), ("XZ", -1), ("XX", 1)])
observables = []
for qa, qb in pairs:
    observables.append([obs1.apply_layout([qa, qb], num_qubits)])
    observables.append([obs2.apply_layout([qa, qb], num_qubits)])

number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)
individual_phases = [[ph] for ph in phases]

# -------------------------Step 2: Optimize problem for quantum hardware execution-------------------------
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
chsh_isa_circuit = pm.run(chsh_circuit)
isa_observables = [
    [o[0].apply_layout(chsh_isa_circuit.layout)] for o in observables
]

# -------------------------Step 3: Execute using Qiskit primitives-------------------------
estimator_hw = Estimator(mode=backend)
estimator_hw.options.environment.job_tags = ["TUT_CI"]

pub = (chsh_isa_circuit, isa_observables, individual_phases)
job = estimator_hw.run(pubs=[pub])
print(f"Job ID: {job.job_id()}")
hw_result = job.result()

# -------------------------Step 4: Post-process and return result in desired classical format-------------------------
# evs has shape (2 * num_pairs, number_of_phases); rows alternate S1, S2
evs = np.asarray(hw_result[0].data.evs)
chsh1_all = evs[0::2]
chsh2_all = evs[1::2]

# A pair "violates" CHSH if its strongest witness exceeds the classical bound
peak = np.maximum(
    np.abs(chsh1_all).max(axis=1), np.abs(chsh2_all).max(axis=1)
)
n_violate = int(np.sum(peak > 2))
print(
    f"{n_violate}/{num_pairs} Bell pairs violated the CHSH inequality "
    f"(mean peak witness {peak.mean():.2f}, classical bound 2)"
)

fig, ax = plt.subplots(figsize=(10, 6))

# Faint individual per-pair curves
for row in chsh1_all:
    ax.plot(phases / np.pi, row, color="#1f77b4", alpha=0.2, lw=1)
for row in chsh2_all:
    ax.plot(phases / np.pi, row, color="#ff7f0e", alpha=0.2, lw=1)

# Bold mean curves across all pairs
ax.plot(
    phases / np.pi,
    chsh1_all.mean(axis=0),
    color="#1f77b4",
    lw=2.5,
    label=r"$\langle S_1 \rangle$ (mean)",
)
ax.plot(
    phases / np.pi,
    chsh2_all.mean(axis=0),
    color="#ff7f0e",
    lw=2.5,
    label=r"$\langle S_2 \rangle$ (mean)",
)

# classical bound +-2 and Tsirelson bound +-2*sqrt(2)
ax.axhline(y=2, color="0.9", linestyle="--")
ax.axhline(y=-2, color="0.9", linestyle="--")
ax.axhline(y=np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.axhline(y=-np.sqrt(2) * 2, color="0.9", linestyle="-.")
ax.fill_between(phases / np.pi, 2, 2 * np.sqrt(2), color="0.6", alpha=0.7)
ax.fill_between(phases / np.pi, -2, -2 * np.sqrt(2), color="0.6", alpha=0.7)

ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))
ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))
ax.set_xlabel(r"$\theta$")
ax.set_ylabel("CHSH witness")
ax.set_title(
    f"CHSH witnesses for {num_pairs} parallel Bell pairs on {backend.name}"
)
ax.legend()
plt.show()

Tiling ibm_pittsburgh with 64 parallel Bell pairs (128 of 156 qubits)
Job ID: d86efd5g7okc73el0rp0
63/64 Bell pairs violated the CHSH inequality (mean peak witness 2.75, classical bound 2)


<Image src="../docs/images/tutorials/chsh-inequality/extracted-outputs/3376bc73-1.avif" alt="Output of the previous code cell" />

The faint curves are the individual Bell pairs and the bold curves are their mean across the device. Every pair traces the same sinusoid predicted by quantum mechanics, and the spread between the faint curves reflects the variation in noise from pair to pair. Wherever a curve enters the grey bands, it has crossed the classical bound of $\pm 2$, and the printed summary confirms that essentially every pair violates the CHSH inequality at the same time.

The peaks fall short of the Tsirelson bound $\pm 2\sqrt{2}$ because of device noise, but the conclusion is unambiguous: the backend sustains genuine entanglement across the whole chip simultaneously, not just on a single hand-picked pair. This is the sense in which the CHSH experiment "scales": not as one larger circuit, but as a parallel benchmark that certifies entanglement everywhere at once.

## Next steps
<Admonition type="tip" title="Recommendations">

If you found this work interesting, you might be interested in the following material:
- [Entanglement in action](/learning/courses/basics-of-quantum-information/entanglement-in-action/chsh-game): a course lesson by John Watrous on Bell states and the CHSH game.
- [Get started with the Estimator primitive](/docs/guides/primitives): a guide on PUBs and parameter sweeps.
- [Real-time benchmarking for qubit selection](/docs/tutorials/real-time-benchmarking-for-qubit-selection): another way to characterize qubit and entanglement quality across a device.
- [`SparsePauliOp` API reference](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp).
</Admonition>